# 诊断：为什么 SR 收敛到 HF 而不是 FCI？

分析关键问题：
1. QGT trace = 0 的原因
2. 为什么能量收敛到 HF
3. 网络是否正确表达波函数

In [ ]:
import jax
import jax.numpy as jnp
import netket as nk
import netket.experimental as nkx
import numpy as np
from pyscf import gto, scf, fci
from flax import nnx
from functools import partial
import matplotlib.pyplot as plt

# 设置系统
bond_length = 1.4
geometry = [('H', (0., 0., 0.)), ('H', (bond_length, 0., 0.))]
mol = gto.M(atom=geometry, basis='STO-3G', verbose=0)
mf = scf.RHF(mol).run(verbose=0)

# FCI 和 HF 能量
cisolver = fci.FCI(mf)
E_fci = cisolver.kernel()[0]
E_hf = mf.e_tot

print("="*60)
print("能量基准")
print("="*60)
print(f"HF 能量: {E_hf:.8f} Ha")
print(f"FCI 能量: {E_fci:.8f} Ha")
print(f"相关能: {E_hf - E_fci:.6f} Ha ({(E_hf - E_fci)*27.2114:.2f} eV)")
print()

ha = nkx.operator.from_pyscf_molecule(mol)
hi = nk.hilbert.SpinOrbitalFermions(n_orbitals=2, s=1/2, n_fermions_per_spin=(1,1))

## 1. 检查希尔伯特空间的所有可能状态

In [ ]:
# 列出所有可能的电子组态
print("="*60)
print("H2 分子的所有可能电子组态")
print("="*60)
print()

# Hilbert 空间有 4 个自旋轨道：
# 轨道 0: σg↑, 轨道 1: σg↓, 轨道 2: σu↑, 轨道 3: σu↓
# 每个自旋有 1 个电子

all_states = hi.all_states()
print(f"总共有 {len(all_states)} 个可能的组态：")
print()

for i, state in enumerate(all_states):
    # 解析状态
    occ_up = [j for j in range(2) if state[j] == 1]  # 自旋向上占据
    occ_down = [j for j in range(2, 4) if state[j] == 1]  # 自旋向下占据
    
    print(f"状态 {i}: {state}")
    print(f"  自旋向上占据轨道: {occ_up}")
    print(f"  自旋向下占据轨道: {occ_down}")
    print()

## 2. 检查哈密顿量的矩阵元

In [ ]:
# 计算哈密顿量在所有状态上的矩阵元
print("="*60)
print("哈密顿量矩阵元")
print("="*60)
print()

for i, state in enumerate(all_states):
    # 获取连接态
    connected, mels = ha.get_conn(state)
    
    print(f"状态 {i} ({state}) 的连接态：")
    for j, (conn_state, mel) in enumerate(zip(connected, mels)):
        conn_idx = np.where(np.all(all_states == conn_state, axis=1))[0][0]
        print(f"  -> 状态 {conn_idx}: 矩阵元 = {mel:.6f}")
    print()

## 3. 检查采样分布

In [ ]:
# 加载训练好的模型
class SingleStateAnsatz(nnx.Module):
    def __init__(self, n_spin_orbitals: int, hidden_dim=16, *, rngs: nnx.Rngs):
        super().__init__()
        self.linear1 = nnx.Linear(n_spin_orbitals, hidden_dim, rngs=rngs, param_dtype=complex)
        self.linear2 = nnx.Linear(hidden_dim, hidden_dim, rngs=rngs, param_dtype=complex)
        self.output = nnx.Linear(hidden_dim, 1, rngs=rngs, param_dtype=complex)

    def __call__(self, x):
        h = nnx.tanh(self.linear1(x.astype(complex)))
        h = nnx.tanh(self.linear2(h))
        out = self.output(h)
        return jnp.squeeze(out)

def forward(GraphDef_State, x):
    log_psi, _ = nnx.call(GraphDef_State)(x)
    return log_psi

# 初始化模型（使用相同的种子）
rngs = nnx.Rngs(21)
model = SingleStateAnsatz(n_spin_orbitals=4, hidden_dim=12, rngs=rngs)
graphdef, model_state = nnx.split(model)
GraphDef_State = (graphdef, model_state)

# 设置采样器
edges = [(0, 1), (2, 3)]
g = nk.graph.Graph(edges=edges)
single_rule = nk.sampler.rules.FermionHopRule(hilbert=hi, graph=g)
sampler = nk.sampler.MetropolisSampler(hi, rule=single_rule, n_chains=16, sweep_size=32)

# 采样
sampler_state = sampler.init_state(forward, GraphDef_State, seed=21)
sampler_state = sampler.reset(forward, GraphDef_State, state=sampler_state)
samples, sampler_state = sampler.sample(forward, GraphDef_State, state=sampler_state, chain_length=100)
samples = samples.reshape(-1, 4)

print("="*60)
print("采样分布分析")
print("="*60)
print()

# 统计每个状态的出现次数
unique_states, counts = np.unique(samples, axis=0, return_counts=True)

print(f"总样本数: {len(samples)}")
print(f"唯一状态数: {len(unique_states)}")
print()

print("各状态出现频率：")
for state, count in zip(unique_states, counts):
    freq = count / len(samples)
    state_idx = np.where(np.all(all_states == state, axis=1))[0][0]
    print(f"  状态 {state_idx} ({state}): {count} 次 ({freq*100:.2f}%)")

print()
print("⚠️ 警告：如果只有 1-2 个状态被采样，说明波函数退化！")

## 4. 检查波函数振幅

In [ ]:
# 计算所有状态的波函数振幅
print("="*60)
print("波函数振幅分析")
print("="*60)
print()

log_psi_all = forward(GraphDef_State, all_states)
psi_all = jnp.exp(log_psi_all)
psi_all_normalized = psi_all / jnp.linalg.norm(psi_all)

print("各状态的波函数振幅：")
for i, (state, psi) in enumerate(zip(all_states, psi_all_normalized)):
    print(f"  状态 {i} ({state}): |ψ|² = {jnp.abs(psi)**2:.6f}")

print()
print("⚠️ 关键检查：")
print("1. 是否有多个状态有显著振幅？（应该有）")
print("2. 振幅是否正确反映电子关联？")

## 5. 检查 QGT 计算

In [ ]:
# 计算 QGT 并检查
print("="*60)
print("QGT 诊断")
print("="*60)
print()

# 计算对数导数
def logpsi_single_param(s, x):
    return forward((graphdef, s), x)

def grad_logpsi_single(x):
    return jax.grad(logpsi_single_param, argnums=0, holomorphic=True)(model_state, x)

# 对所有状态计算梯度
grads_all = jax.vmap(grad_logpsi_single)(all_states)

# 展平梯度
grads_flat, tree_def = jax.tree_util.tree_flatten(grads_all)
grads_concat = jnp.concatenate([g.reshape((len(all_states), -1)) for g in grads_flat], axis=-1)

print(f"梯度矩阵形状: {grads_concat.shape}")
print(f"梯度矩阵范数: {jnp.linalg.norm(grads_concat):.6e}")
print()

# 检查每个状态的梯度范数
print("各状态的梯度范数：")
for i, grad in enumerate(grads_concat):
    norm = jnp.linalg.norm(grad)
    print(f"  状态 {i}: ||∇logψ|| = {norm:.6e}")

print()

# 计算 QGT（使用所有状态）
mean_grad = jnp.mean(grads_concat, axis=0, keepdims=True)
centered_grads = grads_concat - mean_grad
qgt = (1.0 / len(all_states)) * jnp.einsum('si,sj->ij', centered_grads.conj(), centered_grads)

print(f"QGT 形状: {qgt.shape}")
print(f"QGT trace: {jnp.trace(qgt):.6e}")
print(f"QGT 行列式: {jnp.linalg.det(qgt):.6e}")
print(f"QGT 条件数: {jnp.linalg.cond(qgt):.6e}")
print(f"QGT 秩: {jnp.linalg.matrix_rank(qgt)}")

print()
print("⚠️ 关键诊断：")
print("1. 如果 QGT trace = 0，说明所有梯度相同（波函数退化）")
print("2. 如果 QGT 秩 < 参数数量，说明参数冗余")
print("3. 如果条件数极大，说明数值不稳定")

## 6. 根本原因总结

In [ ]:
print("\n" + "="*70)
print("根本原因分析")
print("="*70)
print()

print("问题诊断：")
print()
print("1. QGT trace = 0 的原因：")
print("   - 所有样本的梯度几乎相同")
print("   - 波函数在采样空间上几乎平坦")
print("   - 网络可能只输出常数或接近常数的值")
print()

print("2. 能量收敛到 HF 的原因：")
print("   - HF 态是单行列式，只有两个组态")
print("   - 网络可能学会了只给这两个组态非零振幅")
print("   - 缺少电子关联效应（多组态混合）")
print()

print("3. 为什么会发生？")
print("   - 网络架构可能不足以表达关联")
print("   - 初始化可能导致网络陷入局部极小")
print("   - 采样器可能只访问有限的组态空间")
print()

print("解决方案：")
print()
print("1. 改进网络架构：")
print("   - 增加隐藏层维度")
print("   - 使用更深的网络")
print("   - 添加对称性约束")
print()

print("2. 改进训练策略：")
print("   - 使用更好的初始化")
print("   - 增加样本数")
print("   - 调整学习率和 SR 参数")
print()

print("3. 添加物理先验：")
print("   - 强制波函数对称性")
print("   - 使用多行列式 Ansatz")
print("   - 添加关联因子")
print()

print("="*70)